In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import csv
import time
import chardet

import os

import PublicDataReader as pdr


D:\seoulmate\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [97]:
# 공간 데이터
space_file = "서울시_행정동_영역.csv"

# 이동 데이터
base_dir = r"D:\seoulmate\서울시_행정동_이동"
month_dirs = {
    "tpss_emd_odps_202408",
    "tpss_emd_odps_202409",
    "tpss_emd_odps_202410",
    "tpss_emd_odps_202411",
    "tpss_emd_odps_202412",
    "tpss_emd_odps_202501",
    "tpss_emd_odps_202502",
    "tpss_emd_odps_202503",
    "tpss_emd_odps_202504",
    "tpss_emd_odps_202505",
    "tpss_emd_odps_202506",
    "tpss_emd_odps_202507",
    "tpss_emd_odps_202508",
    "tpss_emd_odps_202509",
    "tpss_emd_odps_202510",
    "tpss_emd_odps_202511",
    "tpss_emd_odps_202512",
    "tpss_emd_odps_202601",
    "tpss_emd_odps_202602",
    "tpss_emd_odps_202603",
}


# 행정동ID-행정동코드-법정동코드 매핑

edm_mapping_name = "서울시_행정동ID_행정동코드_맵핑.csv"


#
# 법정동 위치크기 + 일단위 행정동 대중교통/버스/지하철 승차수
#
# 법정동코드|법정동이름|행정동ID|행정동이름|AREA_M2|LAT|LON|전체승객수|지하철승객수|버스승객수
spatiotemporal_name = "서울시_행정동_시공간_202401_202603.csv"



In [98]:
space_df = pd.read_csv(space_file, encoding="utf-8-sig")

print(f"행정동코드 갯수:{len(space_df['adm_cd2'].unique())}")
space_df.head(10)

행정동코드 갯수:426


,adm_cd2,adm_nm,AREA_M2,LAT,LON
0,1111053000,서울특별시 종로구 사직동,1165780,126.97014,37.57411
1,1111054000,서울특별시 종로구 삼청동,1361445,126.98111,37.58801
2,1111055000,서울특별시 종로구 부암동,2202073,126.96256,37.59670
3,1111056000,서울특별시 종로구 평창동,9017316,126.96927,37.61396
4,1111057000,서울특별시 종로구 무악동,467575,126.95899,37.57774
5,1111058000,서울특별시 종로구 교남동,345990,126.96416,37.57105
6,1111060000,서울특별시 종로구 가회동,605383,126.98662,37.58268
7,1111061500,서울특별시 종로구 종로1·2·3·4가동,2394166,126.98973,37.57508
8,1111063000,서울특별시 종로구 종로5·6가동,633463,127.00424,37.57302
9,1111064000,서울특별시 종로구 이화동,755902,127.00307,37.57969


In [99]:

# 법정동 갯수는 467개
# 행정동 갯수는 426개

edm_mapping_df = pd.read_csv(edm_mapping_name, encoding="utf-8-sig")


print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}")

edm_mapping_df.head(2)

맵핑 갯수:426, 행정동ID 갯수:426, 행정동코드 갯수:426


,행정동_ID,행정동코드,행정동이름
0,11010720,1111051500,청운효자동
1,11010530,1111053000,사직동


In [100]:
# 1:1 인지 확인

#
# edm_mapping_df["cnt"] = edm_mapping_df.groupby("행정동_ID")["행정동코드"].transform("count")
#
# print(len(edm_mapping_df))
#
# dup_df = edm_mapping_df[edm_mapping_df["cnt"]!=1]
#
# print(len(dup_df))
#
# dup_df


In [101]:


# 기존 spatiotemporal_name 삭제
if os.path.exists(spatiotemporal_name):
    os.remove(spatiotemporal_name)


# spatiotemporal_name 생성
for month_dir in sorted(month_dirs):

    passenger_dir = os.path.join(base_dir, month_dir)

    files = os.listdir(passenger_dir)

    for file in files:

        passenger_file = os.path.join(passenger_dir, file)

        print(f"{passenger_file} 처리")

        try:
            passenger_df = pd.read_csv(passenger_file,encoding="cp949")
        except:
            passenger_df = pd.read_csv(passenger_file,encoding="utf-8-sig")

        #print(passenger_df.head(10))

        # 시작을 모르면 제외
        passenger_df = passenger_df[passenger_df["시작_행정동_ID"]!=-1]

        #
        # 시작 행정동기준으로 합친다
        #
        passenger_sum_df = passenger_df.groupby(["기준_날짜","시작_행정동_ID"],as_index=False)[
            ["전체_승객_수 (명)","지하철_승객_수 (명)","버스_승객_수 (명)"]
        ].sum()

        #
        # print(f"갯수:{len(passenger_sum_df)}")
        # print(passenger_sum_df.head(2))


        # 행정동코드와 합치기
        spatiotemporal_df = pd.merge(
            edm_mapping_df,
            passenger_sum_df,
            left_on="행정동_ID",
            right_on="시작_행정동_ID",
            how="right"
        )

        # # 컬럼 이름 변경
        # print(f"갯수:{len(spatiotemporal_df)}")
        # print(spatiotemporal_df.head(2))


        # 영역과 합치기
        spatiotemporal_df = pd.merge(
            space_df,
            spatiotemporal_df,
            left_on="adm_cd2",
            right_on="행정동코드",
            how="right"
        )


        # int64
        cols = ["행정동코드","행정동_ID","AREA_M2"]
        spatiotemporal_df[cols] = spatiotemporal_df[cols].astype("Int64")



        # # 컬럼 이름 변경
        # print(f"갯수:{len(spatiotemporal_df)}")
        # print(spatiotemporal_df.head(2))

        spatiotemporal_df.rename(columns={
            "전체_승객_수 (명)":"전체승객수",
            "지하철_승객_수 (명)":"지하철승객수",
            "버스_승객_수 (명)":"버스승객수"
                        })[["기준_날짜",
                           "행정동코드",
                           "행정동이름",
                           "행정동_ID",
                           "AREA_M2",
                           "LAT",
                           "LON",
                           "전체승객수",
                           "지하철승객수",
                           "버스승객수"]].to_csv(
            spatiotemporal_name,
            index=False,
            header=not os.path.exists(spatiotemporal_name),
            mode="a",
            encoding="utf-8-sig")



D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240801.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240802.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240803.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240804.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240805.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240806.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240807.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240808.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240809.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240810.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240811.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240812.csv 처리
D:\seoulmate\서울시_행정동_이동\tpss_emd_odps_202408\tpss_emd_odps_20240813.csv 처리
D:\seoulmate\서울시_행정동_이동\t

In [7]:
# 월단위를 생성

monthly_spatiotemporal_df = pd.read_csv("서울시_행정동_시공간_일단위_202408_202603.csv", encoding="utf-8-sig")


monthly_spatiotemporal_df["YYYYMM"] = pd.to_datetime(
    monthly_spatiotemporal_df["기준_날짜"], format="%Y%m%d").dt.strftime("%Y%m")

index_cols = [
    "YYYYMM",
    "행정동코드"
]

sum_cols = [
    "전체승객수",
    "지하철승객수",
    "버스승객수"
]


monthly_spatiotemporal_df = monthly_spatiotemporal_df.groupby(index_cols,as_index=False).agg({
        "행정동이름": "first",
        "AREA_M2": "first",
        "LAT": "first",
        "LON": "first",
        "전체승객수": "sum",
        "지하철승객수": "sum",
        "버스승객수": "sum",
    })


monthly_spatiotemporal_df.to_csv("서울시_행정동_시공간_월단위_202408_202603.csv", index=False, encoding="utf-8-sig")


print(f"갯수: {len(monthly_spatiotemporal_df)}")
monthly_spatiotemporal_df.head(2)


갯수: 8513


,YYYYMM,행정동코드,행정동이름,AREA_M2,LAT,LON,전체승객수,지하철승객수,버스승객수
0,202408,1.111052e+09,청운효자동,2438307.0,126.97042,37.58466,252717,0,252717
1,202408,1.111053e+09,사직동,1165780.0,126.97014,37.57411,2375818,1557165,818653


In [4]:
#

s_df = pd.read_csv("../data/서울시_행정동_시공간_월단위_202408_202603.csv", encoding="utf-8-sig")


s_df[["YYYYMM","행정동코드","행정동이름","AREA_M2","LAT","LON","전체승객수"]].to_csv(
    "../data/서울시_행정동_시공간_월단위_202408_202603_base.csv", index=False, encoding="utf-8-sig")




